<a href="https://colab.research.google.com/github/badabheem/Biological_SEQ/blob/main/RNA_Seq_Str_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup ColabFold

In [104]:
import sys
import os
import shutil
import requests
import tarfile
import subprocess

# Check if NVIDIA GPU is available
if not shutil.which("nvidia-smi"):
  print("WARNING: CUDA not detected! ColabFold will be slow.")

In [105]:
# Install ColabFold and its dependencies
print("Upgrading pip...")
!pip install -q --upgrade pip

print("Installing build-essential and python3-dev for C extensions...")
!apt-get update -qq && apt-get install -y build-essential python3-dev

print("Installing compatible setuptools (required by torch)....")
!pip install -q setuptools==65.5.1 # Or any version < 82

print("Installing ColabFold and its dependencies...")
!pip install -q colabfold[alphafold-minus-jax]

print("ColabFold installation complete.")

Upgrading pip...
Installing build-essential and python3-dev for C extensions...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
python3-dev is already the newest version (3.10.6-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 124 not upgraded.
Installing compatible setuptools (required by torch)....
Installing ColabFold and its dependencies...
ColabFold installation complete.


## RNA to Protein Translation

In [106]:
from Bio.Seq import Seq
from Bio.Data import CodonTable

def translate_rna_to_protein(rna_sequence):
    """Translates an RNA sequence to a protein sequence.
    If a stop codon (*) is encountered, the sequence is truncated at that point.
    """
    # Use the standard genetic code
    # Note: RNA sequences are usually represented as 'U' (uracil) instead of 'T' (thymine)
    # The Seq object can handle this for translation.
    messenger_rna = Seq(rna_sequence.replace('U', 'T')) # BioPython expects DNA alphabet for translation initially, then converts.

    # Default to standard genetic code
    try:
        protein_sequence_obj = messenger_rna.translate(table="Standard")
        protein_sequence_str = str(protein_sequence_obj)

        # Truncate at the first stop codon if found
        if '*' in protein_sequence_str:
            protein_sequence_str = protein_sequence_str.split('*')[0]

        # Ensure the sequence is not empty after truncation
        if not protein_sequence_str:
            print(f"Warning: Translation of {rna_sequence} resulted in an empty protein sequence after truncating stop codons.")
            return None

        return protein_sequence_str
    except Exception as e:
        print(f"Error during translation of {rna_sequence}: {e}")
        return None

# The 5 nucleotide sequences provided by the user
nucleotide_sequences = [
    "AGAAGGGACUCACGA",
    "GAUAGGGACUGUGAU",
    "GAUUGGGACUCUUCA",
    "UCUAAGGACUACUCG",
    "GCAGAGGACUAUCCA"
]

protein_sequences = []
for rna_seq in nucleotide_sequences:
    protein_seq = translate_rna_to_protein(rna_seq)
    if protein_seq:
        protein_sequences.append(protein_seq)

print("Translated Protein Sequences:")
for i, seq in enumerate(protein_sequences):
    print(f"Sequence {i+1}: {seq}")

Translated Protein Sequences:
Sequence 1: RRDSR
Sequence 2: DRDCD
Sequence 3: DWDSS
Sequence 4: SKDYS
Sequence 5: AEDYP


## Re-processing New RNA Sequences with Corrected Translation

## Re-running Processing of New RNA Sequences with Corrected Translation

In [107]:
# Define the new set of nucleotide sequences provided by the user (re-loading to ensure fresh start)
nucleotide_sequences = [
    "UACCUGGACUCAGCA",
    "CAGCGGAACUCUACC",
    "UGAUGGAACUUAUAG",
    "CAAAUGGACAGCAAA",
    "GCAAAGAACAACUAC"
]

print("New Nucleotide Sequences re-loaded for re-processing.")

New Nucleotide Sequences re-loaded for re-processing.


In [108]:
# This cell has been modified to store valid (protein_sequence, original_nucleotide_sequence) pairs
# and to correctly populate `protein_sequences` and `nucleotide_sequences` for subsequent steps.
original_nucleotide_sequences_input = [
    "UACCUGGACUCAGCA",
    "CAGCGGAACUCUACC",
    "UGAUGGAACUUAUAG",
    "CAAAUGGACAGCAAA",
    "GCAAAGAACAACUAC"
]

valid_protein_nucleotide_pairs = []
protein_sequences_for_colabfold = []

print("Translating new RNA sequences (with updated stop codon handling):")
for original_rna_seq in original_nucleotide_sequences_input:
    protein_seq = translate_rna_to_protein(original_rna_seq)
    if protein_seq:
        valid_protein_nucleotide_pairs.append((protein_seq, original_rna_seq))
        protein_sequences_for_colabfold.append(protein_seq)
    else:
        print(f"Skipped RNA sequence '{original_rna_seq}' due to empty protein translation.")

print("Translated Protein Sequences for ColabFold (valid entries only):")
for i, (protein_seq, nucleotide_seq) in enumerate(valid_protein_nucleotide_pairs):
    print(f"Sequence {i+1} (RNA: {nucleotide_seq}): {protein_seq}")

# Update global variables with valid sequences for consistency
protein_sequences = protein_sequences_for_colabfold
nucleotide_sequences = [pair[1] for pair in valid_protein_nucleotide_pairs]

# Ensure `output_dir` is defined for subsequent cells if this is the first execution path
output_dir = "./prediction_results_new_sequences" # Re-define if not in scope

Translating new RNA sequences (with updated stop codon handling):
Skipped RNA sequence 'UGAUGGAACUUAUAG' due to empty protein translation.
Translated Protein Sequences for ColabFold (valid entries only):
Sequence 1 (RNA: UACCUGGACUCAGCA): YLDSA
Sequence 2 (RNA: CAGCGGAACUCUACC): QRNST
Sequence 3 (RNA: CAAAUGGACAGCAAA): QMDSK
Sequence 4 (RNA: GCAAAGAACAACUAC): AKNNY


In [109]:
import os
from colabfold.batch import run
import colabfold.download

# Explicitly download ColabFold AlphaFold parameters to ensure they are available.
print("Explicitly downloading ColabFold AlphaFold parameters...")
colabfold.download.download_alphafold_params(model_type='alphafold2_ptm')
print("ColabFold AlphaFold parameters download complete.")

# Define output directory for new sequences
output_dir = "./prediction_results_new_sequences"

# Create output directory if it doesn't exist
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# Prepare queries for ColabFold with the valid protein sequences
queries = []
for i, (protein_seq, _) in enumerate(valid_protein_nucleotide_pairs): # Use valid_protein_nucleotide_pairs
    jobname = f"new_protein_{i+1}" # jobname will now correctly reflect index of *predicted* proteins
    queries.append((jobname, protein_seq, None, None))

# ColabFold prediction parameters (reusing previous settings)
num_recycles = 3
model_type = "auto"
stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

print(f"Starting ColabFold prediction for {len(queries)} new sequences...")
print(f"Output directory: {output_dir}")

results = run(
    queries=queries,
    result_dir=output_dir,
    num_recycles=num_recycles,
    model_type=model_type,
    stop_at_score=stop_at_score,
    num_models=5,
    is_complex=False # Set to False for individual protein predictions
)

print("ColabFold prediction complete for new sequences.")

Explicitly downloading ColabFold AlphaFold parameters...
ColabFold AlphaFold parameters download complete.
Starting ColabFold prediction for 4 new sequences...
Output directory: ./prediction_results_new_sequences
ColabFold prediction complete for new sequences.


### Visualization of Re-Predicted Structures

In [110]:
import py3Dmol

# Iterate through the actual number of predicted proteins
for i in range(len(valid_protein_nucleotide_pairs)): # Use len of the valid pairs
    protein_number = i + 1 # protein_number will be 1, 2, 3, 4
    protein_pdb_path = f"{output_dir}/new_protein_{protein_number}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"

    try:
        with open(protein_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying New Protein {protein_number}:")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for new_protein_{protein_number}: {protein_pdb_path}")
    except Exception as e:
        print(f"Error displaying new_protein_{protein_number}: {e}")

Displaying New Protein 1:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 2:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 3:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 4:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Re-Calculating Average pLDDT Score for New Predictions

In [111]:
import json

new_plddt_scores = []

# Iterate through each predicted protein based on valid_protein_nucleotide_pairs count
for i in range(len(valid_protein_nucleotide_pairs)):
    protein_number = i + 1 # protein_number will be 1, 2, 3, 4
    score_file_path = f"{output_dir}/new_protein_{protein_number}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json"

    try:
        with open(score_file_path, 'r') as f:
            data = json.load(f)
            if 'plddt' in data:
                new_plddt_scores.extend(data['plddt'])
            else:
                print(f"pLDDT score not found in {score_file_path}")
    except FileNotFoundError:
        print(f"Score file not found: {score_file_path}")
    except Exception as e:
        print(f"Error processing {score_file_path}: {e}")

if new_plddt_scores:
    average_new_plddt = sum(new_plddt_scores) / len(new_plddt_scores)
    print(f"All individual pLDDT scores for new sequences: {new_plddt_scores}")
    print(f"Average pLDDT score for all new {len(valid_protein_nucleotide_pairs)} proteins: {average_new_plddt:.2f}")
else:
    print("No pLDDT scores were found or processed for new sequences.")

All individual pLDDT scores for new sequences: [67.5, 76.94, 70.31, 71.81, 71.44, 71.06, 82.38, 82.56, 76.5, 69.62, 70.06, 78.88, 77.69, 78.56, 70.75, 70.94, 76.19, 72.31, 72.25, 72.62]
Average pLDDT score for all new 4 proteins: 74.02


### Re-Saving and Renaming New PDB Files

In [112]:
import os
import shutil

# Define the new folder for renamed PDB files for the new sequences
output_renamed_pdb_dir_new = './renamed_pdb_structures_new_sequences'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir_new):
    os.makedirs(output_renamed_pdb_dir_new)
    print(f"Created directory: {output_renamed_pdb_dir_new}")

# Iterate through the successfully translated protein sequences and their original nucleotide sequences
for i, (protein_seq, nucleotide_seq) in enumerate(valid_protein_nucleotide_pairs):
    protein_number = i + 1 # Corresponds to new_protein_1, new_protein_2, etc. from ColabFold
    # Construct the original PDB file path from the new prediction_results directory
    original_pdb_filename = f"new_protein_{protein_number}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
    original_pdb_path = os.path.join(output_dir, original_pdb_filename)

    # Define the new PDB file name using the correct original nucleotide sequence
    new_pdb_filename = f"{nucleotide_seq}.pdb"
    new_pdb_path = os.path.join(output_renamed_pdb_dir_new, new_pdb_filename)

    try:
        # Copy and rename the file
        shutil.copy(original_pdb_path, new_pdb_path)
        print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
    except FileNotFoundError:
        print(f"Original PDB file not found for new_protein {protein_number} ({nucleotide_seq}): {original_pdb_path}")
    except Exception as e:
        print(f"Error processing new_protein {protein_number} ({nucleotide_seq}): {e}")

print("All new PDB files have been processed and saved in their respective new folder.")

Copied and renamed 'new_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/UACCUGGACUCAGCA.pdb'
Copied and renamed 'new_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAGCGGAACUCUACC.pdb'
Copied and renamed 'new_protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAAAUGGACAGCAAA.pdb'
Copied and renamed 'new_protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/GCAAAGAACAACUAC.pdb'
All new PDB files have been processed and saved in their respective new folder.


In [113]:
# Define the raw set of nucleotide sequences provided by the user
# This list will remain untouched as the original input.
original_nucleotide_sequences_raw = [
    "UACCUGGACUCAGCA",
    "CAGCGGAACUCUACC",
    "UGAUGGAACUUAUAG",
    "CAAAUGGACAGCAAA",
    "GCAAAGAACAACUAC"
]

print("Raw Nucleotide Sequences re-loaded.")

Raw Nucleotide Sequences re-loaded.


In [114]:
# Define the new set of nucleotide sequences provided by the user (raw input)
original_nucleotide_sequences_raw = [
    "UACCUGGACUCAGCA",
    "CAGCGGAACUCUACC",
    "UGAUGGAACUUAUAG",
    "CAAAUGGACAGCAAA",
    "GCAAAGAACAACUAC"
]

# This list will store tuples of (protein_sequence, original_nucleotide_sequence)
# for sequences that successfully translate into non-empty proteins.
valid_protein_nucleotide_pairs = []

print("Translating new RNA sequences (with stop codon handling):")
for original_rna_seq in original_nucleotide_sequences_raw:
    protein_seq = translate_rna_to_protein(original_rna_seq)
    if protein_seq:
        valid_protein_nucleotide_pairs.append((protein_seq, original_rna_seq))
    else:
        print(f"Skipped RNA sequence '{original_rna_seq}' due to empty protein translation.")

# CRITICAL: Populate the global 'protein_sequences' and 'nucleotide_sequences'
# *directly* from the filtered valid_protein_nucleotide_pairs list.
protein_sequences = [pair[0] for pair in valid_protein_nucleotide_pairs]
nucleotide_sequences = [pair[1] for pair in valid_protein_nucleotide_pairs]

print("Translated Protein Sequences for ColabFold (valid entries only):")
for i, (protein_seq, original_rna_seq) in enumerate(valid_protein_nucleotide_pairs):
    print(f"Sequence {i+1} (RNA: {original_rna_seq}): {protein_seq}")

# Ensure `output_dir` is defined for subsequent cells if this is the first execution path
output_dir = "./prediction_results_new_sequences" # Re-define if not in scope, for robustness

Translating new RNA sequences (with stop codon handling):
Skipped RNA sequence 'UGAUGGAACUUAUAG' due to empty protein translation.
Translated Protein Sequences for ColabFold (valid entries only):
Sequence 1 (RNA: UACCUGGACUCAGCA): YLDSA
Sequence 2 (RNA: CAGCGGAACUCUACC): QRNST
Sequence 3 (RNA: CAAAUGGACAGCAAA): QMDSK
Sequence 4 (RNA: GCAAAGAACAACUAC): AKNNY


In [115]:
import os
from colabfold.batch import run

# Define output directory for new sequences
output_dir = "./prediction_results_new_sequences"

# Create output directory if it doesn't exist
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# Prepare queries for ColabFold with the valid protein sequences from 'protein_sequences'
queries = []
for i, protein_seq in enumerate(protein_sequences): # Use the globally updated protein_sequences
    jobname = f"new_protein_{i+1}" # jobname will now correctly reflect index of *predicted* proteins
    queries.append((jobname, protein_seq, None, None))

# ColabFold prediction parameters (reusing previous settings)
num_recycles = 3
model_type = "auto"
stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

print(f"Starting ColabFold prediction for {len(queries)} new sequences...")
print(f"Output directory: {output_dir}")

results = run(
    queries=queries,
    result_dir=output_dir,
    num_recycles=num_recycles,
    model_type=model_type,
    stop_at_score=stop_at_score,
    num_models=5,
    is_complex=False # Set to False for individual protein predictions
)

print("ColabFold prediction complete for new sequences.")

Starting ColabFold prediction for 4 new sequences...
Output directory: ./prediction_results_new_sequences
ColabFold prediction complete for new sequences.


### Visualization of Re-Predicted Structures

In [116]:
import py3Dmol

# Iterate through the actual number of predicted proteins
for i in range(len(protein_sequences)): # Use len of the globally updated protein_sequences
    protein_number = i + 1 # protein_number will be 1, 2, 3, 4
    protein_pdb_path = f"{output_dir}/new_protein_{protein_number}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"

    try:
        with open(protein_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying New Protein {protein_number}:")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for new_protein_{protein_number}: {protein_pdb_path}")
    except Exception as e:
        print(f"Error displaying new_protein_{protein_number}: {e}")

Displaying New Protein 1:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 2:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 3:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 4:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Re-Calculating Average pLDDT Score for New Predictions

In [117]:
import json

new_plddt_scores = []

# Iterate through each predicted protein based on the globally updated protein_sequences count
for i in range(len(protein_sequences)):
    protein_number = i + 1 # protein_number will be 1, 2, 3, 4
    score_file_path = f"{output_dir}/new_protein_{protein_number}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json"

    try:
        with open(score_file_path, 'r') as f:
            data = json.load(f)
            if 'plddt' in data:
                new_plddt_scores.extend(data['plddt'])
            else:
                print(f"pLDDT score not found in {score_file_path}")
    except FileNotFoundError:
        print(f"Score file not found: {score_file_path}")
    except Exception as e:
        print(f"Error processing {score_file_path}: {e}")

if new_plddt_scores:
    average_new_plddt = sum(new_plddt_scores) / len(new_plddt_scores)
    print(f"All individual pLDDT scores for new sequences: {new_plddt_scores}")
    print(f"Average pLDDT score for all new {len(protein_sequences)} proteins: {average_new_plddt:.2f}")
else:
    print("No pLDDT scores were found or processed for new sequences.")

All individual pLDDT scores for new sequences: [67.5, 76.94, 70.31, 71.81, 71.44, 71.06, 82.38, 82.56, 76.5, 69.62, 70.06, 78.88, 77.69, 78.56, 70.75, 70.94, 76.19, 72.31, 72.25, 72.62]
Average pLDDT score for all new 4 proteins: 74.02


### Re-Saving and Renaming New PDB Files

In [118]:
import os
import shutil

# Define the new folder for renamed PDB files for the new sequences
output_renamed_pdb_dir_new = './renamed_pdb_structures_new_sequences'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir_new):
    os.makedirs(output_renamed_pdb_dir_new)
    print(f"Created directory: {output_renamed_pdb_dir_new}")

# Iterate through the successfully translated protein sequences and their original nucleotide sequences
for i, (protein_seq, nucleotide_seq) in enumerate(valid_protein_nucleotide_pairs):
    protein_number = i + 1 # Corresponds to new_protein_1, new_protein_2, etc. from ColabFold
    # Construct the original PDB file path from the new prediction_results directory
    original_pdb_filename = f"new_protein_{protein_number}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
    original_pdb_path = os.path.join(output_dir, original_pdb_filename)

    # Define the new PDB file name using the correct original nucleotide sequence
    new_pdb_filename = f"{nucleotide_seq}.pdb"
    new_pdb_path = os.path.join(output_renamed_pdb_dir_new, new_pdb_filename)

    try:
        # Copy and rename the file
        shutil.copy(original_pdb_path, new_pdb_path)
        print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
    except FileNotFoundError:
        print(f"Original PDB file not found for new_protein {protein_number} ({nucleotide_seq}): {original_pdb_path}")
    except Exception as e:
        print(f"Error processing new_protein {protein_number} ({nucleotide_seq}): {e}")

print("All new PDB files have been processed and saved in their respective new folder.")

Copied and renamed 'new_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/UACCUGGACUCAGCA.pdb'
Copied and renamed 'new_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAGCGGAACUCUACC.pdb'
Copied and renamed 'new_protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAAAUGGACAGCAAA.pdb'
Copied and renamed 'new_protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/GCAAAGAACAACUAC.pdb'
All new PDB files have been processed and saved in their respective new folder.


## ColabFold Prediction

In [119]:
print("Downloading ColabFold models...")
# The colabfold.download module contains functions to download models.
# For a standard prediction, it usually downloads all required models.
# This step ensures all necessary model files are present in the cache.

# NOTE: ColabFold handles model downloads implicitly in the 'run' function if not found,
# however, sometimes it fails silently or due to specific environment issues.
# Explicitly importing and calling a download function is a safeguard.

# As of recent ColabFold versions, the 'run' function is robust enough to download models,
# but if persistent FileNotFoundError occurs, ensure dependencies or cache are fine.
# The error indicates that the model file was not found by numpy.load within colabfold.
# Let's try to ensure the model path is correctly handled by colabfold.
# Re-running the cell is the next step to allow colabfold to handle the download.
# I will remove the download cell for now and just execute the run cell again. If it fails, I will add the download models cell.

In [120]:
import colabfold.download

print("Explicitly downloading ColabFold AlphaFold parameters...")
# This function ensures all AlphaFold parameters are downloaded and cached.
# It will download them only if they are not already present.
colabfold.download.download_alphafold_params(model_type='alphafold2_ptm')
print("ColabFold AlphaFold parameters download complete.")

Explicitly downloading ColabFold AlphaFold parameters...
ColabFold AlphaFold parameters download complete.


In [121]:
import os
from colabfold.batch import run

# Define output directory
output_dir = "./prediction_results"

# Create output directory if it doesn't exist
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# Prepare queries for ColabFold
# The run function expects a list of (jobname, sequence, optional_a3m, optional_a3m_paired) tuples
# For single sequence prediction without precomputed MSA, we provide None for the last two elements.
queries = []
for i, seq in enumerate(protein_sequences):
    jobname = f"protein_{i+1}"
    # Append (jobname, sequence, None, None) to match the expected 4-element structure
    queries.append((jobname, seq, None, None))

# ColabFold prediction parameters
# You can adjust these based on your needs
num_recycles = 3
model_type = "auto"
stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

print(f"Starting ColabFold prediction for {len(queries)} sequences...")
print(f"Output directory: {output_dir}")

# Run ColabFold
results = run(
    queries=queries,
    result_dir=output_dir,
    num_recycles=num_recycles,
    model_type=model_type,
    stop_at_score=stop_at_score,
    num_models=5,
    is_complex=False # Set to False for individual protein predictions
)

print("ColabFold prediction complete.")

Starting ColabFold prediction for 4 sequences...
Output directory: ./prediction_results
ColabFold prediction complete.


In [122]:
import py3Dmol

# Path to the PDB file for protein_1
protein_1_pdb_path = "./prediction_results/protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"

# Read the PDB file content
with open(protein_1_pdb_path, 'r') as f:
    pdb_data = f.read()

# Create a 3Dmol viewer
view = py3Dmol.view(width=800, height=600)

# Add the protein model to the viewer
view.addModel(pdb_data, 'pdb')

# Apply a cartoon representation and color it by residue property (e.g., secondary structure)
view.setStyle({'cartoon': {'color': 'spectrum'}})

# Zoom to fit the model in the view
view.zoomTo()

# Display the viewer
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [123]:
import json

plddt_scores = []

# Iterate through each protein from 1 to 5
for i in range(1, 6):
    # Construct the path to the score JSON file
    score_file_path = f"./prediction_results/protein_{i}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json"

    try:
        with open(score_file_path, 'r') as f:
            data = json.load(f)
            # Extract the plddt scores (which is a list of scores for multiple models)
            if 'plddt' in data:
                # Extend plddt_scores with the list of scores from data['plddt']
                plddt_scores.extend(data['plddt'])
            else:
                print(f"pLDDT score not found in {score_file_path}")
    except FileNotFoundError:
        print(f"Score file not found: {score_file_path}")
    except Exception as e:
        print(f"Error processing {score_file_path}: {e}")

if plddt_scores:
    average_plddt = sum(plddt_scores) / len(plddt_scores)
    print(f"All individual pLDDT scores: {plddt_scores}")
    print(f"Average pLDDT score for all 5 proteins: {average_plddt:.2f}")
else:
    print("No pLDDT scores were found or processed.")

Score file not found: ./prediction_results/protein_5_scores_rank_001_alphafold2_ptm_model_1_seed_000.json
All individual pLDDT scores: [67.5, 76.94, 70.31, 71.81, 71.44, 71.06, 82.38, 82.56, 76.5, 69.62, 70.06, 78.88, 77.69, 78.56, 70.75, 70.94, 76.19, 72.31, 72.25, 72.62]
Average pLDDT score for all 5 proteins: 74.02


In [124]:
import py3Dmol

for i in range(1, 6):
    protein_pdb_path = f"./prediction_results/protein_{i}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"

    try:
        with open(protein_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying Protein {i}:")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for protein_{i}: {protein_pdb_path}")
    except Exception as e:
        print(f"Error displaying protein_{i}: {e}")

Displaying Protein 1:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying Protein 2:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying Protein 3:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying Protein 4:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

PDB file not found for protein_5: ./prediction_results/protein_5_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb


In [125]:
import os
import shutil

# Define the new folder for renamed PDB files
output_renamed_pdb_dir = './renamed_pdb_structures'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir):
    os.makedirs(output_renamed_pdb_dir)
    print(f"Created directory: {output_renamed_pdb_dir}")

# Iterate through the translated protein sequences and original nucleotide sequences
for i, (protein_seq, nucleotide_seq) in enumerate(zip(protein_sequences, nucleotide_sequences)):
    # Construct the original PDB file path from prediction_results
    original_pdb_filename = f"protein_{i+1}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
    original_pdb_path = os.path.join("./prediction_results", original_pdb_filename)

    # Define the new PDB file name using the nucleotide sequence
    # Replace any characters that might be invalid in a filename (e.g., U, A, G, C) with empty strings or underscores if necessary.
    # For simplicity, we'll assume the nucleotide sequences themselves are valid enough for filenames.
    new_pdb_filename = f"{nucleotide_seq}.pdb"
    new_pdb_path = os.path.join(output_renamed_pdb_dir, new_pdb_filename)

    try:
        # Copy and rename the file
        shutil.copy(original_pdb_path, new_pdb_path)
        print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
    except FileNotFoundError:
        print(f"Original PDB file not found for protein {i+1}: {original_pdb_path}")
    except Exception as e:
        print(f"Error processing protein {i+1} ({nucleotide_seq}): {e}")

print("All PDB files have been processed and saved in the new folder.")

Copied and renamed 'protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures/UACCUGGACUCAGCA.pdb'
Copied and renamed 'protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures/CAGCGGAACUCUACC.pdb'
Copied and renamed 'protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures/CAAAUGGACAGCAAA.pdb'
Copied and renamed 'protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures/GCAAAGAACAACUAC.pdb'
All PDB files have been processed and saved in the new folder.


## Processing New RNA Sequences

In [126]:
# Define the new set of nucleotide sequences provided by the user
nucleotide_sequences = [
    "UACCUGGACUCAGCA",
    "CAGCGGAACUCUACC",
    "UGAUGGAACUUAUAG",
    "CAAAUGGACAGCAAA",
    "GCAAAGAACAACUAC"
]

print("New Nucleotide Sequences loaded.")

New Nucleotide Sequences loaded.


In [127]:
protein_sequences = []
for rna_seq in nucleotide_sequences:
    protein_seq = translate_rna_to_protein(rna_seq)
    if protein_seq:
        protein_sequences.append(protein_seq)

print("Translated Protein Sequences for the new input:")
for i, seq in enumerate(protein_sequences):
    print(f"Sequence {i+1}: {seq}")

Translated Protein Sequences for the new input:
Sequence 1: YLDSA
Sequence 2: QRNST
Sequence 3: QMDSK
Sequence 4: AKNNY


In [128]:
import os
from colabfold.batch import run

# Define output directory (can be the same or different, creating if not exists)
output_dir = "./prediction_results_new_sequences"

# Create output directory if it doesn't exist
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# Prepare queries for ColabFold with the new protein sequences
queries = []
for i, seq in enumerate(protein_sequences):
    jobname = f"new_protein_{i+1}"
    queries.append((jobname, seq, None, None))

# ColabFold prediction parameters (reusing previous settings)
num_recycles = 3
model_type = "auto"
stop_at_score = 0.85

print(f"Starting ColabFold prediction for {len(queries)} new sequences...")
print(f"Output directory: {output_dir}")

results = run(
    queries=queries,
    result_dir=output_dir,
    num_recycles=num_recycles,
    model_type=model_type,
    stop_at_score=stop_at_score,
    num_models=5,
    is_complex=False
)

print("ColabFold prediction complete for new sequences.")

Starting ColabFold prediction for 4 new sequences...
Output directory: ./prediction_results_new_sequences
ColabFold prediction complete for new sequences.


### Visualization of New Predicted Structures

In [129]:
import py3Dmol

for i in range(1, len(protein_sequences) + 1):
    protein_pdb_path = f"{output_dir}/new_protein_{i}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"

    try:
        with open(protein_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying New Protein {i}:")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for new_protein_{i}: {protein_pdb_path}")
    except Exception as e:
        print(f"Error displaying new_protein_{i}: {e}")

Displaying New Protein 1:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 2:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 3:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying New Protein 4:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Calculating Average pLDDT Score for New Predictions

In [130]:
import json

new_plddt_scores = []

# Iterate through each protein for the new sequences
for i in range(1, len(protein_sequences) + 1):
    score_file_path = f"{output_dir}/new_protein_{i}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json"

    try:
        with open(score_file_path, 'r') as f:
            data = json.load(f)
            if 'plddt' in data:
                new_plddt_scores.extend(data['plddt'])
            else:
                print(f"pLDDT score not found in {score_file_path}")
    except FileNotFoundError:
        print(f"Score file not found: {score_file_path}")
    except Exception as e:
        print(f"Error processing {score_file_path}: {e}")

if new_plddt_scores:
    average_new_plddt = sum(new_plddt_scores) / len(new_plddt_scores)
    print(f"All individual pLDDT scores for new sequences: {new_plddt_scores}")
    print(f"Average pLDDT score for all new {len(protein_sequences)} proteins: {average_new_plddt:.2f}")
else:
    print("No pLDDT scores were found or processed for new sequences.")

All individual pLDDT scores for new sequences: [67.5, 76.94, 70.31, 71.81, 71.44, 71.06, 82.38, 82.56, 76.5, 69.62, 70.06, 78.88, 77.69, 78.56, 70.75, 70.94, 76.19, 72.31, 72.25, 72.62]
Average pLDDT score for all new 4 proteins: 74.02


### Saving and Renaming New PDB Files

In [131]:
import os
import shutil

# Define the new folder for renamed PDB files for the new sequences
output_renamed_pdb_dir_new = './renamed_pdb_structures_new_sequences'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir_new):
    os.makedirs(output_renamed_pdb_dir_new)
    print(f"Created directory: {output_renamed_pdb_dir_new}")

# Iterate through the translated protein sequences and original nucleotide sequences
for i, (protein_seq, nucleotide_seq) in enumerate(zip(protein_sequences, nucleotide_sequences)):
    # Construct the original PDB file path from the new prediction_results directory
    original_pdb_filename = f"new_protein_{i+1}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
    original_pdb_path = os.path.join(output_dir, original_pdb_filename)

    # Define the new PDB file name using the nucleotide sequence
    new_pdb_filename = f"{nucleotide_seq}.pdb"
    new_pdb_path = os.path.join(output_renamed_pdb_dir_new, new_pdb_filename)

    try:
        # Copy and rename the file
        shutil.copy(original_pdb_path, new_pdb_path)
        print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
    except FileNotFoundError:
        print(f"Original PDB file not found for new_protein {i+1}: {original_pdb_path}")
    except Exception as e:
        print(f"Error processing new_protein {i+1} ({nucleotide_seq}): {e}")

print("All new PDB files have been processed and saved in their respective new folder.")

Copied and renamed 'new_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/UACCUGGACUCAGCA.pdb'
Copied and renamed 'new_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAGCGGAACUCUACC.pdb'
Copied and renamed 'new_protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/UGAUGGAACUUAUAG.pdb'
Copied and renamed 'new_protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_new_sequences/CAAAUGGACAGCAAA.pdb'
All new PDB files have been processed and saved in their respective new folder.


## Modeling a Specific Modified RNA Sequence with (m6A)

In [132]:
# New modified RNA sequence provided by the user
modified_rna_sequence_input = "AGAAGGG(m6A)CUCACGA"

# Clean the RNA sequence for translation by replacing (m6A) with A.
# The (m6A) modification does not change the coding for the amino acid.
cleaned_rna_for_translation = modified_rna_sequence_input.replace('(m6A)', 'A')

print(f"Original modified RNA sequence: {modified_rna_sequence_input}")
print(f"RNA sequence for translation: {cleaned_rna_for_translation}")

# Translate the cleaned RNA to protein using the previously defined function
specific_protein_seq = translate_rna_to_protein(cleaned_rna_for_translation)

if specific_protein_seq:
    print(f"Translated protein sequence: {specific_protein_seq}")
else:
    print("Translation failed or resulted in an empty protein sequence.")

Original modified RNA sequence: AGAAGGG(m6A)CUCACGA
RNA sequence for translation: AGAAGGGACUCACGA
Translated protein sequence: RRDSR


In [133]:
import os
from colabfold.batch import run

# Ensure AlphaFold parameters are downloaded, if not already present.
# This is a safeguard, as colabfold.batch.run typically handles this, but explicit check ensures robustness.
import colabfold.download
colabfold.download.download_alphafold_params(model_type='alphafold2_ptm')

# Define a specific output directory for this modified sequence to keep results separate
specific_output_dir = "./prediction_results_m6A_modified_rna"

# Create output directory if it doesn't exist
if not os.path.isdir(specific_output_dir):
    os.makedirs(specific_output_dir)
    print(f"Created directory: {specific_output_dir}")

# Prepare query for ColabFold for this single protein sequence
jobname_for_modified_rna = "m6A_modified_protein"

if specific_protein_seq:
    queries_for_modified_rna = [(jobname_for_modified_rna, specific_protein_seq, None, None)]

    # ColabFold prediction parameters (reusing previous settings)
    num_recycles = 3
    model_type = "auto"
    stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

    print(f"Starting ColabFold prediction for the modified RNA sequence protein ({specific_protein_seq})...")
    print(f"Output directory: {specific_output_dir}")

    results_modified_rna = run(
        queries=queries_for_modified_rna,
        result_dir=specific_output_dir,
        num_recycles=num_recycles,
        model_type=model_type,
        stop_at_score=stop_at_score,
        num_models=5,
        is_complex=False # Set to False for individual protein predictions
    )

    print("ColabFold prediction complete for the modified RNA sequence protein.")
else:
    print("ColabFold prediction skipped: No valid protein sequence to predict.")

Starting ColabFold prediction for the modified RNA sequence protein (RRDSR)...
Output directory: ./prediction_results_m6A_modified_rna
ColabFold prediction complete for the modified RNA sequence protein.


### Visualization of the Predicted Structure for the Modified RNA Sequence

In [134]:
import py3Dmol
import os

# Ensure specific_protein_seq and jobname_for_modified_rna are available from previous cells
# (re-defining for robustness if running cells out of order)
# specific_output_dir = "./prediction_results_m6A_modified_rna"
# jobname_for_modified_rna = "m6A_modified_protein"

if 'specific_protein_seq' in locals() and specific_protein_seq:
    predicted_pdb_path = os.path.join(specific_output_dir, f"{jobname_for_modified_rna}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

    try:
        with open(predicted_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying predicted structure for {jobname_for_modified_rna}:")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for {jobname_for_modified_rna}: {predicted_pdb_path}")
    except Exception as e:
        print(f"Error displaying structure for {jobname_for_modified_rna}: {e}")
else:
    print("Cannot visualize: no protein sequence translated or variable not defined.")

Displaying predicted structure for m6A_modified_protein:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Calculating Average pLDDT Score for the Modified RNA Sequence

In [135]:
import json
import os

# Ensure specific_protein_seq and jobname_for_modified_rna are available from previous cells
# (re-defining for robustness if running cells out of order)
# specific_output_dir = "./prediction_results_m6A_modified_rna"
# jobname_for_modified_rna = "m6A_modified_protein"

if 'specific_protein_seq' in locals() and specific_protein_seq:
    score_file_path_modified_rna = os.path.join(specific_output_dir, f"{jobname_for_modified_rna}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json")
    plddt_scores_modified_rna = []

    try:
        with open(score_file_path_modified_rna, 'r') as f:
            data = json.load(f)
            if 'plddt' in data:
                plddt_scores_modified_rna.extend(data['plddt'])
            else:
                print(f"pLDDT score not found in {score_file_path_modified_rna}")
    except FileNotFoundError:
        print(f"Score file not found: {score_file_path_modified_rna}")
    except Exception as e:
        print(f"Error processing {score_file_path_modified_rna}: {e}")

    if plddt_scores_modified_rna:
        average_plddt_modified_rna = sum(plddt_scores_modified_rna) / len(plddt_scores_modified_rna)
        print(f"Individual pLDDT scores for {jobname_for_modified_rna}: {plddt_scores_modified_rna}")
        print(f"Average pLDDT score for {jobname_for_modified_rna}: {average_plddt_modified_rna:.2f}")
    else:
        print("No pLDDT scores were found or processed for the modified RNA sequence protein.")
else:
    print("Cannot calculate pLDDT score: no protein sequence translated or variable not defined.")

Individual pLDDT scores for m6A_modified_protein: [63.59, 61.88, 65.19, 73.5, 67.94]
Average pLDDT score for m6A_modified_protein: 66.42


### Saving and Renaming the PDB File for the Modified RNA Sequence

In [136]:
import os
import shutil

# Ensure modified_rna_sequence_input and jobname_for_modified_rna are available
# (re-defining for robustness if running cells out of order)
# specific_output_dir = "./prediction_results_m6A_modified_rna"
# modified_rna_sequence_input = "AGAAGGG(m6A)CUCACGA"
# jobname_for_modified_rna = "m6A_modified_protein"

if 'specific_protein_seq' in locals() and specific_protein_seq:
    output_renamed_pdb_dir_m6A = './renamed_pdb_structures_m6A_modified_rna'

    # Create the new folder if it doesn't exist
    if not os.path.isdir(output_renamed_pdb_dir_m6A):
        os.makedirs(output_renamed_pdb_dir_m6A)
        print(f"Created directory: {output_renamed_pdb_dir_m6A}")

    original_pdb_filename = f"{jobname_for_modified_rna}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
    original_pdb_path = os.path.join(specific_output_dir, original_pdb_filename)

    # Define the new PDB file name using the original modified RNA sequence.
    # Replace '(m6A)' with 'm6A' to create a valid and descriptive filename.
    new_pdb_filename_m6A = f"{modified_rna_sequence_input.replace('(m6A)', 'm6A')}.pdb"
    new_pdb_path_m6A = os.path.join(output_renamed_pdb_dir_m6A, new_pdb_filename_m6A)

    try:
        shutil.copy(original_pdb_path, new_pdb_path_m6A)
        print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path_m6A}'")
    except FileNotFoundError:
        print(f"Original PDB file not found for {jobname_for_modified_rna}: {original_pdb_path}")
    except Exception as e:
        print(f"Error processing {jobname_for_modified_rna}: {e}")
else:
    print("Cannot save PDB file: no protein sequence translated or variable not defined.")

Copied and renamed 'm6A_modified_protein_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_m6A_modified_rna/AGAAGGGm6ACUCACGA.pdb'


## Modeling Additional Modified RNA Sequences with (m6A)

In [137]:
# New modified RNA sequences provided by the user
additional_modified_rna_sequences = [
    "GAUAGGG(m6A)CUGUGAU",
    "GAUUGGG(m6A)CUCUUCA",
    "UCUAAGG(m6A)CUACUCG",
    "GCAGAGG(m6A)CUAUCCA"
]

# Prepare to store cleaned RNA and translated protein sequences
cleaned_rna_for_translation_list = []
additional_valid_protein_nucleotide_pairs = []
additional_protein_sequences_for_colabfold = []

print("Processing additional modified RNA sequences...")
for original_modified_rna_seq in additional_modified_rna_sequences:
    # Clean the RNA sequence for translation by replacing (m6A) with A.
    cleaned_rna = original_modified_rna_seq.replace('(m6A)', 'A')
    cleaned_rna_for_translation_list.append(cleaned_rna)

    print(f"Original RNA: {original_modified_rna_seq} -> Cleaned RNA for translation: {cleaned_rna}")

    # Translate the cleaned RNA to protein
    protein_seq = translate_rna_to_protein(cleaned_rna)

    if protein_seq:
        additional_valid_protein_nucleotide_pairs.append((protein_seq, original_modified_rna_seq))
        additional_protein_sequences_for_colabfold.append(protein_seq)
        print(f"  Translated protein sequence: {protein_seq}")
    else:
        print(f"  Translation failed or resulted in an empty protein sequence for {original_modified_rna_seq}.")

print("\nTranslated Protein Sequences for additional modified RNA (valid entries only):")
for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs):
    print(f"Sequence {i+1} (RNA: {original_rna_seq}): {protein_seq}")

Processing additional modified RNA sequences...
Original RNA: GAUAGGG(m6A)CUGUGAU -> Cleaned RNA for translation: GAUAGGGACUGUGAU
  Translated protein sequence: DRDCD
Original RNA: GAUUGGG(m6A)CUCUUCA -> Cleaned RNA for translation: GAUUGGGACUCUUCA
  Translated protein sequence: DWDSS
Original RNA: UCUAAGG(m6A)CUACUCG -> Cleaned RNA for translation: UCUAAGGACUACUCG
  Translated protein sequence: SKDYS
Original RNA: GCAGAGG(m6A)CUAUCCA -> Cleaned RNA for translation: GCAGAGGACUAUCCA
  Translated protein sequence: AEDYP

Translated Protein Sequences for additional modified RNA (valid entries only):
Sequence 1 (RNA: GAUAGGG(m6A)CUGUGAU): DRDCD
Sequence 2 (RNA: GAUUGGG(m6A)CUCUUCA): DWDSS
Sequence 3 (RNA: UCUAAGG(m6A)CUACUCG): SKDYS
Sequence 4 (RNA: GCAGAGG(m6A)CUAUCCA): AEDYP


In [138]:
import os
from colabfold.batch import run

# Define a specific output directory for these additional modified sequences
additional_output_dir = "./prediction_results_additional_m6A_rna"

# Create output directory if it doesn't exist
if not os.path.isdir(additional_output_dir):
    os.makedirs(additional_output_dir)
    print(f"Created directory: {additional_output_dir}")

# Prepare queries for ColabFold for these protein sequences
additional_queries = []
if additional_protein_sequences_for_colabfold:
    for i, protein_seq in enumerate(additional_protein_sequences_for_colabfold):
        # Use a jobname that reflects its origin and index
        jobname = f"additional_m6A_protein_{i+1}"
        additional_queries.append((jobname, protein_seq, None, None))

    # ColabFold prediction parameters (reusing previous settings)
    num_recycles = 3
    model_type = "auto"
    stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

    print(f"\nStarting ColabFold prediction for {len(additional_queries)} additional modified RNA sequences...")
    print(f"Output directory: {additional_output_dir}")

    results_additional_m6A = run(
        queries=additional_queries,
        result_dir=additional_output_dir,
        num_recycles=num_recycles,
        model_type=model_type,
        stop_at_score=stop_at_score,
        num_models=5,
        is_complex=False # Set to False for individual protein predictions
    )

    print("ColabFold prediction complete for additional modified RNA sequences.")
else:
    print("ColabFold prediction skipped: No valid protein sequences to predict from additional RNA.")


Starting ColabFold prediction for 4 additional modified RNA sequences...
Output directory: ./prediction_results_additional_m6A_rna
ColabFold prediction complete for additional modified RNA sequences.


### Visualization of Predicted Structures for Additional Modified RNA Sequences

In [139]:
import py3Dmol
import os

# Ensure additional_valid_protein_nucleotide_pairs and additional_output_dir are available

if additional_valid_protein_nucleotide_pairs:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs):
        jobname = f"additional_m6A_protein_{i+1}"
        predicted_pdb_path = os.path.join(additional_output_dir, f"{jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

        try:
            with open(predicted_pdb_path, 'r') as f:
                pdb_data = f.read()

            print(f"Displaying predicted structure for {original_rna_seq} (Protein {i+1}):")
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.zoomTo()
            view.show()
        except FileNotFoundError:
            print(f"PDB file not found for {original_rna_seq}: {predicted_pdb_path}")
        except Exception as e:
            print(f"Error displaying structure for {original_rna_seq}: {e}")
else:
    print("Cannot visualize: no valid protein sequences to display.")

Displaying predicted structure for GAUAGGG(m6A)CUGUGAU (Protein 1):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for GAUUGGG(m6A)CUCUUCA (Protein 2):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for UCUAAGG(m6A)CUACUCG (Protein 3):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for GCAGAGG(m6A)CUAUCCA (Protein 4):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Calculating Average pLDDT Scores for Additional Modified RNA Sequences

In [140]:
import json
import os

additional_plddt_scores_combined = []

if additional_valid_protein_nucleotide_pairs:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs):
        jobname = f"additional_m6A_protein_{i+1}"
        score_file_path = os.path.join(additional_output_dir, f"{jobname}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json")
        protein_plddt_scores = []

        try:
            with open(score_file_path, 'r') as f:
                data = json.load(f)
                if 'plddt' in data:
                    protein_plddt_scores.extend(data['plddt'])
                    additional_plddt_scores_combined.extend(data['plddt'])
                else:
                    print(f"pLDDT score not found in {score_file_path}")
        except FileNotFoundError:
            print(f"Score file not found: {score_file_path}")
        except Exception as e:
            print(f"Error processing {score_file_path}: {e}")

        if protein_plddt_scores:
            average_plddt_for_protein = sum(protein_plddt_scores) / len(protein_plddt_scores)
            print(f"Average pLDDT score for {original_rna_seq}: {average_plddt_for_protein:.2f}")
        else:
            print(f"No pLDDT scores were found or processed for {original_rna_seq}.")

    if additional_plddt_scores_combined:
        overall_average_plddt = sum(additional_plddt_scores_combined) / len(additional_plddt_scores_combined)
        print(f"\nOverall average pLDDT score for all {len(additional_valid_protein_nucleotide_pairs)} additional proteins: {overall_average_plddt:.2f}")
    else:
        print("No pLDDT scores were found or processed for any of the additional proteins.")
else:
    print("Cannot calculate pLDDT scores: no valid protein sequences processed.")

Average pLDDT score for GAUAGGG(m6A)CUGUGAU: 77.43
Average pLDDT score for GAUUGGG(m6A)CUCUUCA: 64.63
Average pLDDT score for UCUAAGG(m6A)CUACUCG: 78.40
Average pLDDT score for GCAGAGG(m6A)CUAUCCA: 76.10

Overall average pLDDT score for all 4 additional proteins: 74.14


### Saving and Renaming PDB Files for Additional Modified RNA Sequences

In [141]:
import os
import shutil

output_renamed_pdb_dir_additional_m6A = './renamed_pdb_structures_additional_m6A_rna'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir_additional_m6A):
    os.makedirs(output_renamed_pdb_dir_additional_m6A)
    print(f"Created directory: {output_renamed_pdb_dir_additional_m6A}")

if additional_valid_protein_nucleotide_pairs:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs):
        jobname = f"additional_m6A_protein_{i+1}"
        original_pdb_filename = f"{jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
        original_pdb_path = os.path.join(additional_output_dir, original_pdb_filename)

        # Define the new PDB file name using the original modified RNA sequence.
        # Replace '(m6A)' with 'm6A' to create a valid and descriptive filename.
        new_pdb_filename = f"{original_rna_seq.replace('(m6A)', 'm6A')}.pdb"
        new_pdb_path = os.path.join(output_renamed_pdb_dir_additional_m6A, new_pdb_filename)

        try:
            shutil.copy(original_pdb_path, new_pdb_path)
            print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
        except FileNotFoundError:
            print(f"Original PDB file not found for {original_rna_seq}: {original_pdb_path}")
        except Exception as e:
            print(f"Error processing {original_rna_seq}: {e}")
else:
    print("Cannot save PDB files: no valid protein sequences processed.")

Copied and renamed 'additional_m6A_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_additional_m6A_rna/GAUAGGGm6ACUGUGAU.pdb'
Copied and renamed 'additional_m6A_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_additional_m6A_rna/GAUUGGGm6ACUCUUCA.pdb'
Copied and renamed 'additional_m6A_protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_additional_m6A_rna/UCUAAGGm6ACUACUCG.pdb'
Copied and renamed 'additional_m6A_protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_additional_m6A_rna/GCAGAGGm6ACUAUCCA.pdb'


## Visualizing a Specific Additional Protein Structure

## Consolidating All m6A Modified PDB Files

In [142]:
import os
import shutil

# Define the new central folder for all m6A modified PDB files
central_m6A_pdb_dir = './all_m6A_renamed_pdb_structures'

# Create the central folder if it doesn't exist
if not os.path.isdir(central_m6A_pdb_dir):
    os.makedirs(central_m6A_pdb_dir)
    print(f"Created central directory: {central_m6A_pdb_dir}")

# Source directories where the renamed PDBs currently reside
source_dirs = [
    './renamed_pdb_structures_m6A_modified_rna', # From the single m6A sequence
    './renamed_pdb_structures_additional_m6A_rna' # From the batch of m6A sequences
]

print("Moving PDB files to the central folder...")
for s_dir in source_dirs:
    if os.path.isdir(s_dir):
        for filename in os.listdir(s_dir):
            if filename.endswith('.pdb'):
                source_path = os.path.join(s_dir, filename)
                destination_path = os.path.join(central_m6A_pdb_dir, filename)
                try:
                    shutil.move(source_path, destination_path)
                    print(f"Moved '{filename}' from '{s_dir}' to '{central_m6A_pdb_dir}'")
                except Exception as e:
                    print(f"Error moving '{filename}': {e}")
    else:
        print(f"Source directory not found: {s_dir}")

print("Consolidation complete. All m6A modified PDB files are now in one folder.")

Moving PDB files to the central folder...
Moved 'AGAAGGGm6ACUCACGA.pdb' from './renamed_pdb_structures_m6A_modified_rna' to './all_m6A_renamed_pdb_structures'
Moved 'GAUUGGGm6ACUCUUCA.pdb' from './renamed_pdb_structures_additional_m6A_rna' to './all_m6A_renamed_pdb_structures'
Moved 'GCAGAGGm6ACUAUCCA.pdb' from './renamed_pdb_structures_additional_m6A_rna' to './all_m6A_renamed_pdb_structures'
Moved 'GAUAGGGm6ACUGUGAU.pdb' from './renamed_pdb_structures_additional_m6A_rna' to './all_m6A_renamed_pdb_structures'
Moved 'UCUAAGGm6ACUACUCG.pdb' from './renamed_pdb_structures_additional_m6A_rna' to './all_m6A_renamed_pdb_structures'
Consolidation complete. All m6A modified PDB files are now in one folder.


In [143]:
import py3Dmol
import os

# Assuming additional_valid_protein_nucleotide_pairs and additional_output_dir are still in scope
# Select the first protein from the list for visualization
if additional_valid_protein_nucleotide_pairs:
    # Get the protein sequence and original RNA sequence for the first protein
    first_protein_seq, first_original_rna_seq = additional_valid_protein_nucleotide_pairs[0]
    first_jobname = "additional_m6A_protein_1" # Corresponding jobname for the first protein

    predicted_pdb_path = os.path.join(additional_output_dir, f"{first_jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

    try:
        with open(predicted_pdb_path, 'r') as f:
            pdb_data = f.read()

        print(f"Displaying predicted structure for {first_original_rna_seq} (Protein 1 - {first_protein_seq}):")
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f"PDB file not found for {first_original_rna_seq}: {predicted_pdb_path}")
    except Exception as e:
        print(f"Error displaying structure for {first_original_rna_seq}: {e}")
else:
    print("No additional protein sequences to visualize.")

Displaying predicted structure for GAUAGGG(m6A)CUGUGAU (Protein 1 - DRDCD):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Modeling Additional Modified RNA Sequences with (m6A) - Batch 2

In [155]:
# New modified RNA sequences provided by the user (Batch 2)
additional_modified_rna_sequences_batch2 = [
    "UACCUGG(m6A)CUCAGCA",
    "CAGCGG(m6A)CUCUACC",
    "UGAUGG(m6A)CUUAUAG",
    "CAAAUGG(m6A)CAGCAAA",
    "GCAAAG(m6A)CAACUAC"
]

# Prepare to store cleaned RNA and translated protein sequences
cleaned_rna_for_translation_list_batch2 = []
additional_valid_protein_nucleotide_pairs_batch2 = []
additional_protein_sequences_for_colabfold_batch2 = []

print("Processing additional modified RNA sequences (Batch 2)...")
for original_modified_rna_seq in additional_modified_rna_sequences_batch2:
    # Clean the RNA sequence for translation by replacing (m6A) with A.
    cleaned_rna = original_modified_rna_seq.replace('(m6A)', 'A')
    cleaned_rna_for_translation_list_batch2.append(cleaned_rna)

    print(f"Original RNA: {original_modified_rna_seq} -> Cleaned RNA for translation: {cleaned_rna}")

    # Translate the cleaned RNA to protein
    protein_seq = translate_rna_to_protein(cleaned_rna)

    if protein_seq:
        additional_valid_protein_nucleotide_pairs_batch2.append((protein_seq, original_modified_rna_seq))
        additional_protein_sequences_for_colabfold_batch2.append(protein_seq)
        print(f"  Translated protein sequence: {protein_seq}")
    else:
        print(f"  Translation failed or resulted in an empty protein sequence for {original_modified_rna_seq}.")

print("\nTranslated Protein Sequences for additional modified RNA (Batch 2 - valid entries only):")
for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs_batch2):
    print(f"Sequence {i+1} (RNA: {original_rna_seq}): {protein_seq}")

Processing additional modified RNA sequences (Batch 2)...
Original RNA: UACCUGG(m6A)CUCAGCA -> Cleaned RNA for translation: UACCUGGACUCAGCA
  Translated protein sequence: YLDSA
Original RNA: CAGCGG(m6A)CUCUACC -> Cleaned RNA for translation: CAGCGGACUCUACC
  Translated protein sequence: QRTL
Original RNA: UGAUGG(m6A)CUUAUAG -> Cleaned RNA for translation: UGAUGGACUUAUAG
  Translation failed or resulted in an empty protein sequence for UGAUGG(m6A)CUUAUAG.
Original RNA: CAAAUGG(m6A)CAGCAAA -> Cleaned RNA for translation: CAAAUGGACAGCAAA
  Translated protein sequence: QMDSK
Original RNA: GCAAAG(m6A)CAACUAC -> Cleaned RNA for translation: GCAAAGACAACUAC
  Translated protein sequence: AKTT

Translated Protein Sequences for additional modified RNA (Batch 2 - valid entries only):
Sequence 1 (RNA: UACCUGG(m6A)CUCAGCA): YLDSA
Sequence 2 (RNA: CAGCGG(m6A)CUCUACC): QRTL
Sequence 3 (RNA: CAAAUGG(m6A)CAGCAAA): QMDSK
Sequence 4 (RNA: GCAAAG(m6A)CAACUAC): AKTT


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [156]:
import os
from colabfold.batch import run

# Define a specific output directory for these additional modified sequences (Batch 2)
additional_output_dir_batch2 = "./prediction_results_m6A_batch2"

# Create output directory if it doesn't exist
if not os.path.isdir(additional_output_dir_batch2):
    os.makedirs(additional_output_dir_batch2)
    print(f"Created directory: {additional_output_dir_batch2}")

# Prepare queries for ColabFold for these protein sequences (Batch 2)
additional_queries_batch2 = []
if additional_protein_sequences_for_colabfold_batch2:
    for i, protein_seq in enumerate(additional_protein_sequences_for_colabfold_batch2):
        # Use a jobname that reflects its origin and index
        jobname = f"m6A_batch2_protein_{i+1}"
        additional_queries_batch2.append((jobname, protein_seq, None, None))

    # ColabFold prediction parameters (reusing previous settings)
    num_recycles = 3
    model_type = "auto"
    stop_at_score = 0.85 # Stop prediction if pLDDT score is above this threshold

    print(f"\nStarting ColabFold prediction for {len(additional_queries_batch2)} additional modified RNA sequences (Batch 2)...")
    print(f"Output directory: {additional_output_dir_batch2}")

    results_additional_m6A_batch2 = run(
        queries=additional_queries_batch2,
        result_dir=additional_output_dir_batch2,
        num_recycles=num_recycles,
        model_type=model_type,
        stop_at_score=stop_at_score,
        num_models=5,
        is_complex=False # Set to False for individual protein predictions
    )

    print("ColabFold prediction complete for additional modified RNA sequences (Batch 2).")
else:
    print("ColabFold prediction skipped: No valid protein sequences to predict from additional RNA (Batch 2).")


Starting ColabFold prediction for 4 additional modified RNA sequences (Batch 2)...
Output directory: ./prediction_results_m6A_batch2
ColabFold prediction complete for additional modified RNA sequences (Batch 2).


In [157]:
import py3Dmol
import os

# Ensure additional_valid_protein_nucleotide_pairs_batch2 and additional_output_dir_batch2 are available

if additional_valid_protein_nucleotide_pairs_batch2:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs_batch2):
        jobname = f"m6A_batch2_protein_{i+1}"
        predicted_pdb_path = os.path.join(additional_output_dir_batch2, f"{jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

        try:
            with open(predicted_pdb_path, 'r') as f:
                pdb_data = f.read()

            print(f"Displaying predicted structure for {original_rna_seq} (Protein {i+1}):")
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.zoomTo()
            view.show()
        except FileNotFoundError:
            print(f"PDB file not found for {original_rna_seq}: {predicted_pdb_path}")
        except Exception as e:
            print(f"Error displaying structure for {original_rna_seq}: {e}")
else:
    print("Cannot visualize: no valid protein sequences to display (Batch 2).")

Displaying predicted structure for UACCUGG(m6A)CUCAGCA (Protein 1):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for CAGCGG(m6A)CUCUACC (Protein 2):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for CAAAUGG(m6A)CAGCAAA (Protein 3):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for GCAAAG(m6A)CAACUAC (Protein 4):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Visualization of Predicted Structures for Additional Modified RNA Sequences - Batch 2

In [148]:
import py3Dmol
import os

# Ensure additional_valid_protein_nucleotide_pairs_batch2 and additional_output_dir_batch2 are available

if additional_valid_protein_nucleotide_pairs_batch2:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs_batch2):
        jobname = f"m6A_batch2_protein_{i+1}"
        predicted_pdb_path = os.path.join(additional_output_dir_batch2, f"{jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

        try:
            with open(predicted_pdb_path, 'r') as f:
                pdb_data = f.read()

            print(f"Displaying predicted structure for {original_rna_seq} (Protein {i+1}):")
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.zoomTo()
            view.show()
        except FileNotFoundError:
            print(f"PDB file not found for {original_rna_seq}: {predicted_pdb_path}")
        except Exception as e:
            print(f"Error displaying structure for {original_rna_seq}: {e}")
else:
    print("Cannot visualize: no valid protein sequences to display (Batch 2).")

Displaying predicted structure for UACCUGG(m6A)CUCAGCA (Protein 1):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for CAGCGG(m6A)CUCUACC (Protein 2):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for CAAAUGG(m6A)CAGCAAA (Protein 3):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Displaying predicted structure for GCAAAG(m6A)CAACUAC (Protein 4):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Calculating Average pLDDT Scores for Additional Modified RNA Sequences - Batch 2

In [158]:
import json
import os

additional_plddt_scores_combined_batch2 = []

if additional_valid_protein_nucleotide_pairs_batch2:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs_batch2):
        jobname = f"m6A_batch2_protein_{i+1}"
        score_file_path = os.path.join(additional_output_dir_batch2, f"{jobname}_scores_rank_001_alphafold2_ptm_model_1_seed_000.json")
        protein_plddt_scores_batch2 = []

        try:
            with open(score_file_path, 'r') as f:
                data = json.load(f)
                if 'plddt' in data:
                    protein_plddt_scores_batch2.extend(data['plddt'])
                    additional_plddt_scores_combined_batch2.extend(data['plddt'])
                else:
                    print(f"pLDDT score not found in {score_file_path}")
        except FileNotFoundError:
            print(f"Score file not found: {score_file_path}")
        except Exception as e:
            print(f"Error processing {score_file_path}: {e}")

        if protein_plddt_scores_batch2:
            average_plddt_for_protein_batch2 = sum(protein_plddt_scores_batch2) / len(protein_plddt_scores_batch2)
            print(f"Average pLDDT score for {original_rna_seq}: {average_plddt_for_protein_batch2:.2f}")
        else:
            print(f"No pLDDT scores were found or processed for {original_rna_seq}.")

    if additional_plddt_scores_combined_batch2:
        overall_average_plddt_batch2 = sum(additional_plddt_scores_combined_batch2) / len(additional_plddt_scores_combined_batch2)
        print(f"\nOverall average pLDDT score for all {len(additional_valid_protein_nucleotide_pairs_batch2)} additional proteins (Batch 2): {overall_average_plddt_batch2:.2f}")
    else:
        print("No pLDDT scores were found or processed for any of the additional proteins (Batch 2).")
else:
    print("Cannot calculate pLDDT scores: no valid protein sequences processed (Batch 2).")

Average pLDDT score for UACCUGG(m6A)CUCAGCA: 71.60
Average pLDDT score for CAGCGG(m6A)CUCUACC: 77.20
Average pLDDT score for CAAAUGG(m6A)CAGCAAA: 75.19
Average pLDDT score for GCAAAG(m6A)CAACUAC: 71.38

Overall average pLDDT score for all 4 additional proteins (Batch 2): 73.79


### Saving and Renaming PDB Files for Additional Modified RNA Sequences - Batch 2

In [159]:
import os
import shutil

output_renamed_pdb_dir_additional_m6A_batch2 = './renamed_pdb_structures_m6A_batch2'

# Create the new folder if it doesn't exist
if not os.path.isdir(output_renamed_pdb_dir_additional_m6A_batch2):
    os.makedirs(output_renamed_pdb_dir_additional_m6A_batch2)
    print(f"Created directory: {output_renamed_pdb_dir_additional_m6A_batch2}")

if additional_valid_protein_nucleotide_pairs_batch2:
    for i, (protein_seq, original_rna_seq) in enumerate(additional_valid_protein_nucleotide_pairs_batch2):
        jobname = f"m6A_batch2_protein_{i+1}"
        original_pdb_filename = f"{jobname}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb"
        original_pdb_path = os.path.join(additional_output_dir_batch2, original_pdb_filename)

        # Define the new PDB file name using the original modified RNA sequence.
        # Replace '(m6A)' with 'm6A' to create a valid and descriptive filename.
        new_pdb_filename = f"{original_rna_seq.replace('(m6A)', 'm6A')}.pdb"
        new_pdb_path = os.path.join(output_renamed_pdb_dir_additional_m6A_batch2, new_pdb_filename)

        try:
            shutil.copy(original_pdb_path, new_pdb_path)
            print(f"Copied and renamed '{original_pdb_filename}' to '{new_pdb_path}'")
        except FileNotFoundError:
            print(f"Original PDB file not found for {original_rna_seq}: {original_pdb_path}")
        except Exception as e:
            print(f"Error processing {original_rna_seq}: {e}")
else:
    print("Cannot save PDB files: no valid protein sequences processed (Batch 2).")

Copied and renamed 'm6A_batch2_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_m6A_batch2/UACCUGGm6ACUCAGCA.pdb'
Copied and renamed 'm6A_batch2_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_m6A_batch2/CAGCGGm6ACUCUACC.pdb'
Copied and renamed 'm6A_batch2_protein_3_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_m6A_batch2/CAAAUGGm6ACAGCAAA.pdb'
Copied and renamed 'm6A_batch2_protein_4_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb' to './renamed_pdb_structures_m6A_batch2/GCAAAGm6ACAACUAC.pdb'


## Consolidating All m6A Modified PDB Files (Including Batch 2)

In [160]:
import os
import shutil

# Define the new central folder for all m6A modified PDB files
central_m6A_pdb_dir = './all_m6A_renamed_pdb_structures'

# Create the central folder if it doesn't exist
if not os.path.isdir(central_m6A_pdb_dir):
    os.makedirs(central_m6A_pdb_dir)
    print(f"Created central directory: {central_m6A_pdb_dir}")

# Source directories where the renamed PDBs currently reside
source_dirs_updated = [
    './renamed_pdb_structures_m6A_modified_rna', # From the single m6A sequence
    './renamed_pdb_structures_additional_m6A_rna', # From the batch of m6A sequences
    './renamed_pdb_structures_m6A_batch2' # From the new batch of m6A sequences
]

print("Moving PDB files to the central folder (updated list)...")
for s_dir in source_dirs_updated:
    if os.path.isdir(s_dir):
        for filename in os.listdir(s_dir):
            if filename.endswith('.pdb'):
                source_path = os.path.join(s_dir, filename)
                destination_path = os.path.join(central_m6A_pdb_dir, filename)
                try:
                    # Use shutil.move to move the file
                    shutil.move(source_path, destination_path)
                    print(f"Moved '{filename}' from '{s_dir}' to '{central_m6A_pdb_dir}'")
                except Exception as e:
                    print(f"Error moving '{filename}': {e}")
    else:
        print(f"Source directory not found: {s_dir}")

print("Consolidation complete. All m6A modified PDB files are now in one central folder.")

Moving PDB files to the central folder (updated list)...
Moved 'UACCUGGm6ACUCAGCA.pdb' from './renamed_pdb_structures_m6A_batch2' to './all_m6A_renamed_pdb_structures'
Moved 'CAGCGGm6ACUCUACC.pdb' from './renamed_pdb_structures_m6A_batch2' to './all_m6A_renamed_pdb_structures'
Moved 'CAAAUGGm6ACAGCAAA.pdb' from './renamed_pdb_structures_m6A_batch2' to './all_m6A_renamed_pdb_structures'
Moved 'GCAAAGm6ACAACUAC.pdb' from './renamed_pdb_structures_m6A_batch2' to './all_m6A_renamed_pdb_structures'
Consolidation complete. All m6A modified PDB files are now in one central folder.


## Visualizing a Specific Additional Protein Structure (Batch 2)

In [152]:
import py3Dmol
import os

# Assuming additional_valid_protein_nucleotide_pairs_batch2 and additional_output_dir_batch2 are still in scope
# Select the first protein from the new batch for visualization
if additional_valid_protein_nucleotide_pairs_batch2:
    # Get the protein sequence and original RNA sequence for the first protein in batch 2
    first_protein_seq_batch2, first_original_rna_seq_batch2 = additional_valid_protein_nucleotide_pairs_batch2[0]
    first_jobname_batch2 = "m6A_batch2_protein_1" # Corresponding jobname for the first protein in batch 2

    predicted_pdb_path_batch2 = os.path.join(additional_output_dir_batch2, f"{first_jobname_batch2}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

    try:
        with open(predicted_pdb_path_batch2, 'r') as f:
            pdb_data_batch2 = f.read()

        print(f"Displaying predicted structure for {first_original_rna_seq_batch2} (Protein 1 - {first_protein_seq_batch2}):")
        view_batch2 = py3Dmol.view(width=800, height=600)
        view_batch2.addModel(pdb_data_batch2, 'pdb')
        view_batch2.setStyle({'cartoon': {'color': 'spectrum'}})
        view_batch2.zoomTo()
        view_batch2.show()
    except FileNotFoundError:
        print(f"PDB file not found for {first_original_rna_seq_batch2}: {predicted_pdb_path_batch2}")
    except Exception as e:
        print(f"Error displaying structure for {first_original_rna_seq_batch2}: {e}")
else:
    print("No additional protein sequences from Batch 2 to visualize.")

Displaying predicted structure for UACCUGG(m6A)CUCAGCA (Protein 1 - YLDSA):


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [154]:
import py3Dmol
import os

# Variables from previous processing of the single m6A sequence
specific_output_dir = "./prediction_results_m6A_modified_rna"
jobname_for_modified_rna = "m6A_modified_protein"
modified_rna_sequence_input = "AGAAGGG(m6A)CUCACGA"

predicted_pdb_path_specific = os.path.join(specific_output_dir, f"{jobname_for_modified_rna}_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb")

try:
    with open(predicted_pdb_path_specific, 'r') as f:
        pdb_data_specific = f.read()

    print(f"Displaying predicted structure for {modified_rna_sequence_input}:")
    view_specific = py3Dmol.view(width=800, height=600)
    view_specific.addModel(pdb_data_specific, 'pdb')
    view_specific.setStyle({'cartoon': {'color': 'spectrum'}})
    view_specific.zoomTo()
    view_specific.show()
except FileNotFoundError:
    print(f"PDB file not found for {modified_rna_sequence_input}: {predicted_pdb_path_specific}")
except Exception as e:
    print(f"Error displaying structure for {modified_rna_sequence_input}: {e}")

Displaying predicted structure for AGAAGGG(m6A)CUCACGA:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [144]:
!cd renamed_pdb_structures_additional_m6A_rna
!ls

additional_m6A_protein_1.a3m
additional_m6A_protein_1_coverage.png
additional_m6A_protein_1.done.txt
additional_m6A_protein_1_env
additional_m6A_protein_1_pae.png
additional_m6A_protein_1_plddt.png
additional_m6A_protein_1_predicted_aligned_error_v1.json
additional_m6A_protein_1_scores_rank_001_alphafold2_ptm_model_1_seed_000.json
additional_m6A_protein_1_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb
additional_m6A_protein_2.a3m
additional_m6A_protein_2_coverage.png
additional_m6A_protein_2.done.txt
additional_m6A_protein_2_env
additional_m6A_protein_2_pae.png
additional_m6A_protein_2_plddt.png
additional_m6A_protein_2_predicted_aligned_error_v1.json
additional_m6A_protein_2_scores_rank_001_alphafold2_ptm_model_1_seed_000.json
additional_m6A_protein_2_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb
additional_m6A_protein_3.a3m
additional_m6A_protein_3_coverage.png
additional_m6A_protein_3.done.txt
additional_m6A_protein_3_env
additional_m6A_protein_3_pae.png
additional_

In [145]:
import shutil

folder_to_zip = './all_m6A_renamed_pdb_structures'
zip_filename = 'all_m6A_renamed_pdb_structures'

print(f"Creating zip archive for '{folder_to_zip}'...")
shutil.make_archive(zip_filename, 'zip', folder_to_zip)
print(f"Archive created: {zip_filename}.zip")
print("You can now download this file from the Colab file browser.")

Creating zip archive for './all_m6A_renamed_pdb_structures'...
Archive created: all_m6A_renamed_pdb_structures.zip
You can now download this file from the Colab file browser.


In [153]:
import os

central_m6A_pdb_dir = './all_m6A_renamed_pdb_structures'

if os.path.isdir(central_m6A_pdb_dir):
    print(f"Files in '{central_m6A_pdb_dir}':")
    for filename in os.listdir(central_m6A_pdb_dir):
        print(filename)
else:
    print(f"Directory '{central_m6A_pdb_dir}' not found.")

Files in './all_m6A_renamed_pdb_structures':
UACCUGGm6ACUCAGCA.pdb
CAGCGGm6ACUCUACC.pdb
GAUUGGGm6ACUCUUCA.pdb
GCAGAGGm6ACUAUCCA.pdb
GAUAGGGm6ACUGUGAU.pdb
UCUAAGGm6ACUACUCG.pdb
CAAAUGGm6ACAGCAAA.pdb
GCAAAGm6ACAACUAC.pdb
AGAAGGGm6ACUCACGA.pdb
